# World Models, JEPA and Recursive Self-Improvement: Three Bets Beyond the Next Token

The runnable code of [World Models, JEPA and Recursive Self-Improvement: Three Bets Beyond the Next Token](https://residual-stream.lvtan71.workers.dev/posts/world-models-jepa-and-self-improvement/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/world-models-jepa-and-self-improvement.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 3 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "world-models-jepa-and-self-improvement"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in []:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## World models: learn the simulator

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/world-models-jepa-and-self-improvement/#world-models-learn-the-simulator)

In [ ]:
import math
import torch
import torch.nn as nn

# A learned simulator, and what happens when you run it on its own output.
# The world: a damped pendulum (angle, angular velocity), stepped 0.05 s at
# a time. A small network learns one step, (state) -> (next state), from
# 20,000 random transitions. Then "imagination": start from real states and
# feed the model its own predictions for H steps, as Dreamer-style agents do
# to plan, and compare with the real trajectory.
wrap = lambda a: (a + math.pi) % (2 * math.pi) - math.pi          # angles live in [-pi, pi)
def step(x, dt=0.05):
    th, om = x[:, 0], x[:, 1]
    om = om + dt * (-9.8 * torch.sin(th) - 0.1 * om)
    return torch.stack([wrap(th + dt * om), om], 1)
def sample_states(n):
    # the training range covers every state the test trajectories reach,
    # so errors below come from compounding, not from leaving the data
    return torch.stack([(torch.rand(n) * 2 - 1) * math.pi, (torch.rand(n) * 2 - 1) * 10], 1)

def train(seed, n=20000, steps=4000):
    torch.manual_seed(seed)
    x = sample_states(n); y = step(x)
    net = nn.Sequential(nn.Linear(2, 128), nn.Tanh(), nn.Linear(128, 128), nn.Tanh(), nn.Linear(128, 2))
    opt = torch.optim.Adam(net.parameters(), 1e-3)
    for _ in range(steps):
        i = torch.randint(0, n, (512,))
        d = torch.stack([wrap(y[i, 0] - x[i, 0]), y[i, 1] - x[i, 1]], 1)
        loss = ((net(x[i]) - d) ** 2).mean()                # predict the change in state
        opt.zero_grad(); loss.backward(); opt.step()
    return lambda s: torch.stack([wrap(s[:, 0] + net(s)[:, 0]), s[:, 1] + net(s)[:, 1]], 1)

horizons = (1, 5, 15, 50, 100)
print("error of imagined angles against the real ones (radians; a random guess would be")
print("off by about 1.8 root-mean-square), 1,000 start states, 3 seeds")
print(f"{'steps imagined':>22s} " + " ".join(f"{h:>8d}" for h in horizons))
errs, meds = torch.zeros(3, len(horizons)), torch.zeros(3, len(horizons))
for seed in range(3):
    model = train(seed)
    with torch.no_grad():
        torch.manual_seed(100)
        # start anywhere with |angle|, |velocity| <= 2.5: some swings, some spins over the top
        real = imag = (torch.rand(1000, 2) * 2 - 1) * 2.5
        h_i = 0
        for h in range(1, max(horizons) + 1):
            real, imag = step(real), model(imag)
            if h in horizons:
                d = wrap(real[:, 0] - imag[:, 0]).abs()
                errs[seed, h_i], meds[seed, h_i] = (d ** 2).mean().sqrt(), d.median(); h_i += 1
for name, v in (("RMS error, seed mean", errs.mean(0)), ("median error, seed mean", meds.mean(0))):
    print(f"{name:>22s} " + " ".join(f"{x:8.4f}" for x in v))
print(f"{'RMS, per seed at 15':>22s} " + " ".join(f"{x:8.4f}" for x in errs[:, 2]))

The post printed:

```text
error of imagined angles against the real ones (radians; a random guess would be
off by about 1.8 root-mean-square), 1,000 start states, 3 seeds
        steps imagined        1        5       15       50      100
  RMS error, seed mean   0.0029   0.0148   0.0493   0.1205   0.2679
median error, seed mean   0.0023   0.0114   0.0387   0.0710   0.1448
   RMS, per seed at 15   0.0247   0.0645   0.0586
```

In [ ]:
# What one Dreamer-style training update imagines. Batch shape and horizon
# follow DreamerV3's defaults: 16 sequences of 64 steps from the replay
# buffer, and 15 imagined steps from every one of those states. Imagination
# runs only the latent dynamics and the actor, critic and reward heads (no
# image encoder or decoder), batched across all start states at once.
B, T, H = 16, 64, 15
starts = B * T
imagined = starts * H
print(f"per update: {starts:,} start states from replay, {imagined:,} imagined steps,")
print(f"computed as {H} sequential batched steps of {starts:,} states each")
# Living through the same number of steps for real, one at a time:
for env_ms, what in ((1, "a fast simulator"), (50, "a robot acting at 20 Hz")):
    print(f"  {imagined:,} real steps in {what} ({env_ms} ms/step): {imagined * env_ms / 1000:,.0f} s")

The post printed:

```text
per update: 1,024 start states from replay, 15,360 imagined steps,
computed as 15 sequential batched steps of 1,024 states each
  15,360 real steps in a fast simulator (1 ms/step): 15 s
  15,360 real steps in a robot acting at 20 Hz (50 ms/step): 768 s
```

## JEPA: predict the meaning, not the pixels

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/world-models-jepa-and-self-improvement/#jepa-predict-the-meaning-not-the-pixels)

In [ ]:
import copy
import torch
import torch.nn as nn

# Joint-embedding prediction on a toy world. Each sample has a hidden state
# s (8 numbers) that a fixed random network renders into 64-number
# "observations"; two views of the same sample differ by heavy noise and a
# random mask (the context sees 50% of the observation, the target another
# draw). An encoder embeds each view; a predictor maps the context's
# embedding to the target's; the loss is their squared distance. Nothing
# in the loss forbids the trivial answer: embed everything as the same
# point. Five variants, three seeds each, against an untrained encoder.
D_S, D_X, D_Z = 8, 64, 32
torch.manual_seed(1234)
render = nn.Sequential(nn.Linear(D_S, 128), nn.Tanh(), nn.Linear(128, D_X)).requires_grad_(False)
def views(n):
    s = torch.randn(n, D_S)
    x = render(s)
    view = lambda: (x + 0.3 * torch.randn_like(x)) * (torch.rand_like(x) < 0.5)
    return s, view(), view()
mlp = lambda i, o: nn.Sequential(nn.Linear(i, 256), nn.ReLU(), nn.Linear(256, o))

def probe(z, s):
    """How much of the hidden state a linear readout recovers from the
    embedding (R^2 on held-out samples): a collapsed embedding scores 0."""
    z = torch.cat([z, torch.ones(len(z), 1)], 1)
    w = torch.linalg.lstsq(z[:2000], s[:2000]).solution
    err = ((z[2000:] @ w - s[2000:]) ** 2).mean()
    return 1 - err / s[2000:].var(0).mean()

def vicreg_terms(z):
    """VICReg's regularizers: keep each embedding dimension's spread across
    the batch above 1, and the dimensions uncorrelated."""
    z = z - z.mean(0)
    std = torch.sqrt(z.var(0) + 1e-4)
    cov = (z.T @ z) / (len(z) - 1)
    off = cov - torch.diag(torch.diag(cov))
    return 25 * torch.relu(1 - std).mean() + (off ** 2).sum() / z.shape[1]

def run(variant, seed, steps=10000):
    torch.manual_seed(seed)
    enc, pred = mlp(D_X, D_Z), mlp(D_Z, D_Z)
    target_enc = copy.deepcopy(enc).requires_grad_(False)
    params = list(enc.parameters()) + (list(pred.parameters()) if variant != "EMA, no predictor" else [])
    opt = torch.optim.Adam(params, 1e-3)
    for _ in range(steps if variant != "untrained encoder" else 0):
        _, ctx, tgt = views(256)
        z_ctx = enc(ctx)
        if variant in ("shared encoder, no tricks", "VICReg regularizers"):
            z_tgt = enc(tgt)                                  # gradients flow into both sides
        elif variant == "stop-gradient":
            z_tgt = enc(tgt).detach()                         # the target side is treated as a constant
        else:                                                 # EMA: a slow copy of the encoder makes targets
            z_tgt = target_enc(tgt)
        p = z_ctx if variant == "EMA, no predictor" else pred(z_ctx)
        loss = ((p - z_tgt) ** 2).sum(1).mean()
        total = loss + (vicreg_terms(z_ctx) + vicreg_terms(z_tgt) if variant == "VICReg regularizers" else 0)
        opt.zero_grad(); total.backward(); opt.step()
        if variant.startswith("EMA"):
            with torch.no_grad():
                for a, b in zip(target_enc.parameters(), enc.parameters()):
                    a.mul_(0.996).add_(0.004 * b)
    with torch.no_grad():
        s, ctx, tgt = views(4000)
        z = enc(ctx)
        last = ((pred(z) - enc(tgt)) ** 2).sum(1).mean().item() if variant == "untrained encoder" else loss.item()
        return last, z.std(0).mean().item(), probe(z, s).item()

print(f"{'variant':28s} {'final loss':>10s} {'embedding std':>14s} {'probe R^2':>10s}   (mean of 3 seeds, 10,000 steps)")
for variant in ("untrained encoder", "shared encoder, no tricks", "stop-gradient", "EMA target + predictor",
                "EMA, no predictor", "VICReg regularizers"):
    r = torch.tensor([run(variant, seed) for seed in range(3)]).mean(0)
    print(f"{variant:28s} {r[0]:10.4f} {r[1]:14.6f} {r[2]:10.2f}", flush=True)

The post printed:

```text
variant                      final loss  embedding std  probe R^2   (mean of 3 seeds, 10,000 steps)
untrained encoder                0.3205       0.055590       0.33
shared encoder, no tricks        0.0000       0.000174       0.07
stop-gradient                    1.3830       0.253997       0.61
EMA target + predictor           1.9868       0.306188       0.61
EMA, no predictor                0.0000       0.000054      -0.00
VICReg regularizers             15.7369       0.927692       0.61
```